In [2]:
import torch
import torch.nn as nn

In [ ]:
class SEBlockPerChannel(nn.Module):
    def __init__(self, height, width, reduction=16):
        super(SEBlockPerChannel, self).__init__()
        self.height = height
        self.width = width
        
        # Fully connected layers to learn the importance of each height (electrode) for each channel
        self.fc1 = nn.Linear(height, height // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(height // reduction, height, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Input x shape: (batch_size, channels, height, width)
        b, c, h, w = x.size()
        
        # Initialize a list to store the reweighted channels
        output = []
        print(f'Shape of X Before SE: {x.size()}')

        # Loop over each channel (for each filter)
        for i in range(c):
            # Extract the i-th channel (filter): shape (batch_size, height, width)
            y = x[:, i, :, :]
            print(f'Shape of X for filter {i} SE: {y.size()}')


            
            # Squeeze: Average pooling along the width (temporal dimension)
            y_squeezed = y.mean(dim=-1)  # shape: (batch_size, height)
            print(f'Shape of X for filter {i} Pooling Layer SE: {y_squeezed.size()}')
            
            # Excitation: Fully connected layers to assign weights to the height dimension
            y_fc1 = self.fc1(y_squeezed)
            print(f'Shape of X for filter {i} FC Layer1 SE: {y_fc1.size()}')
            y_relu = self.relu(y_fc1)

            y_fc2 = self.fc2(y_relu)
            print(f'Shape of X for filter {i} FC Layer2 SE: {y_fc2.size()}')
            y_sigmoid = self.sigmoid(y_fc2)  # shape: (batch_size, height)
            
            # Reshape to (batch_size, 1, height, 1) to broadcast
            y_sigmoid = y_sigmoid.view(b, 1, h, 1)
            print(f'Shape of X for filter {i} Sigmoid SE: {y_sigmoid.size()}')
            
            # Scale the original channel data
            y_scaled = y.unsqueeze(1) * y_sigmoid.expand_as(y.unsqueeze(1))  # Reshape y to (batch_size, 1, height, width)
            print(f'Shape of X for filter {i} After SE: {y_scaled.size()}')
            
            # Append to output list
            output.append(y_scaled)
        
        # Concatenate along the channel dimension to restore the original shape
        output = torch.cat(output, dim=1)  # shape: (batch_size, channels, height, width)
        print(f'Shape of X for After SE: {output.size()}')
        
        return output


In [3]:
# Define SEBlockHeightWidth (as before)
class SEBlockHeightWidth(nn.Module):
    def __init__(self, height, width, reduction=16):
        super(SEBlockHeightWidth, self).__init__()
        self.height = height
        self.width = width

        # Fully connected layers to learn the importance of each height (electrode)
        self.fc1 = nn.Linear(height, height // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(height // reduction, height, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Input x shape: (batch_size, channels, height, width)
        b, c, h, w = x.size()
        print(f'Shape of X Before SE: {x.size()}')


        # Squeeze: Average pooling along the width (temporal dimension)
        y = x.mean(dim=-1)  # shape: (batch_size, channels, height)
        print(f'Shape of X Pooling Layer SE: {y.size()}')


        # Excitation: Fully connected layers to assign weights to the height dimension
        y = self.fc1(y)
        print(f'Shape of X FC Layer1 SE: {y.size()}')

        y = self.relu(y)
        

        y = self.fc2(y)
        print(f'Shape of X FC Layer2 SE: {y.size()}')
        
        y = self.sigmoid(y)  # shape: (batch_size, channels, height)

        # Reshape and scale: Reshape y to (batch_size, channels, height, 1)
        y = y.view(b, c, h, 1)
        print(f'Shape of X Reshaped Layer SE: {y.size()}')

        # Scale the original input x with learned weights (broadcast across width)
        return x * y.expand_as(x)  # element-wise multiplication

# Define the model incorporating SEBlock
class EEGNetWithSE(nn.Module):
    def __init__(self):
        super(EEGNetWithSE, self).__init__()
        
        # First convolutional layer
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(1, 64), padding="same")
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        
        # SE block for electrode (height) weighting
        self.se_block = SEBlockPerChannel(height=27, width=2000, reduction=3)
        
        # Fully connected layer for classification (example)
        self.fc = nn.Linear(16 * 27 * 2000, 2)  # Modify based on your task

    def forward(self, x):
        # Apply convolutional layer
        print(f'Shape of X: {x.size()}')
        x = self.conv1(x)
        print(f'Shape of X after conv2d: {x.size()}')
        
        x = self.bn1(x)
        print(f'Shape of X after BN: {x.size()}')

        x = self.relu(x)

        
        # Apply SE block to reweight the height dimension
        x = self.se_block(x)
        print(f'Shape of X after SE: {x.size()}')

        
        # Flatten the data for classification
        x = x.view(x.size(0), -1)  # Flatten all dimensions for fully connected layer
        x = self.fc(x)
        
        return x




In [4]:
# Define a dummy EEG dataset (shape: batch_size x height x width)
# Assuming you have batch_size = 10 samples
batch_size = 10
eeg_data = torch.randn(batch_size, 27, 2000)  # Random data for example

# Reshape to match CNN input: batch_size x channels x height x width
eeg_data = eeg_data.unsqueeze(1)  # Add channel dimension, new shape: (batch_size, 1, 27, 2000)

# Instantiate the model
model = EEGNetWithSE()

# Forward pass: pass the EEG data through the model
output = model(eeg_data)

# Output shape after the forward pass
print("Output shape:", output.shape)


Shape of X: torch.Size([10, 1, 27, 2000])


d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\modules\conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,


Shape of X after conv2d: torch.Size([10, 16, 27, 2000])
Shape of X after BN: torch.Size([10, 16, 27, 2000])
Shape of X Before SE: torch.Size([10, 16, 27, 2000])
Shape of X Pooling Layer SE: torch.Size([10, 16, 27])
Shape of X FC Layer1 SE: torch.Size([10, 16, 3])
Shape of X FC Layer2 SE: torch.Size([10, 16, 27])
Shape of X Reshaped Layer SE: torch.Size([10, 16, 27, 1])
Shape of X after SE: torch.Size([10, 16, 27, 2000])
Output shape: torch.Size([10, 10])
